<a href="https://colab.research.google.com/github/wilsonjunior-fc/projeto-final-IA-ml-comex-brasil/blob/main/Projeto_Final_ML_Comex_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Final — Aprendizado de Máquina
## Estimativa do Valor FOB em Operações de Comércio Exterior do Brasil

**Integrantes:** [NOME 1], [NOME 2], [NOME 3]

**Fonte dos dados:** [Data of Brazilian Import and Export data — Kaggle](https://www.kaggle.com/datasets/juniorfazzio/data-of-brazilian-import-and-export-data), compilado a partir do [Comex Stat / MDIC](http://comexstat.mdic.gov.br/pt/home).

Este notebook segue a estrutura exigida pelo roteiro da disciplina (seções 5.1 a 5.7).


## 0. Configuração do ambiente
Instala dependências e baixa o dataset diretamente do Kaggle (sem depender de arquivo local), conforme exigência do roteiro (seção 4).

In [1]:
# Instala o kagglehub (client oficial do Kaggle, mais simples que a API antiga)
!pip install kagglehub -q

import kagglehub
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [2]:
# Autenticação com a API do Kaggle usando o Secrets do Colab (mais seguro que colar o token direto no notebook).
#
# Configuração (uma única vez, por pessoa que for rodar o notebook):
#   1. No Kaggle: Settings > API > "Generate New Token" -> copiar o token gerado.
#   2. No Colab: clicar no icone de chave (Secrets) na barra lateral esquerda.
#   3. Criar um secret chamado KAGGLE_API_TOKEN, colar o token, e ativar "Notebook access".
#
# NUNCA colar o token diretamente no codigo abaixo -- isso vaza a chave para quem tiver acesso
# ao notebook (inclusive no GitHub do grupo).

from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

# Faz o download do dataset (banco SQLite completo, ~3.43 GB) para o ambiente do Colab.
path = kagglehub.dataset_download("juniorfazzio/data-of-brazilian-import-and-export-data")
print("Dataset baixado em:", path)

for f in os.listdir(path):
    print(f)


100%|██████████| 1.46G/1.46G [00:14<00:00, 109MB/s]

Extracting files...


Dataset baixado em: /root/.cache/kagglehub/datasets/juniorfazzio/data-of-brazilian-import-and-export-data/versions/1
database.db


In [ ]:
# Conecta no banco e localiza o arquivo .db (nome pode variar levemente, ex: database.db)
db_file = [f for f in os.listdir(path) if f.endswith(".db")][0]
db_path = os.path.join(path, db_file)
print("Arquivo do banco:", db_path)

conn = sqlite3.connect(db_path)


### ⚠️ Passo obrigatório: inspecionar as tabelas reais do banco
O dataset tem 8 tabelas (confirmado na página do Kaggle), mas os nomes exatos e como elas se relacionam
(tabelas de fatos EXP/IMP + tabelas de apoio) precisam ser confirmados aqui antes de seguir.

In [ ]:
# Lista todas as tabelas do banco
tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tabelas)


In [ ]:
# Para cada tabela, mostra as colunas e os tipos (PRAGMA table_info)
for nome_tabela in tabelas["name"]:
    print(f"\n=== {nome_tabela} ===")
    info = pd.read_sql_query(f"PRAGMA table_info({nome_tabela});", conn)
    print(info[["name", "type"]])


In [ ]:
# Nomes reais das tabelas confirmados na inspecao do banco:
TABELA_EXPORTACAO = "exp_completa"
TABELA_IMPORTACAO = "imp_completa"

# Dado o tamanho do banco (3.43 GB), carregamos uma amostra filtrada por periodo
# em vez do dataset inteiro. Ajustar o filtro conforme a capacidade de RAM disponivel.
FILTRO_ANO = "CO_ANO >= 2018"

query_exp = f"SELECT * FROM {TABELA_EXPORTACAO} WHERE {FILTRO_ANO};"
query_imp = f"SELECT * FROM {TABELA_IMPORTACAO} WHERE {FILTRO_ANO};"

df_exp = pd.read_sql_query(query_exp, conn)
df_imp = pd.read_sql_query(query_imp, conn)
df_exp["TIPO_OPERACAO"] = "Exportacao"
df_imp["TIPO_OPERACAO"] = "Importacao"

df = pd.concat([df_exp, df_imp], ignore_index=True)
print(df.shape)
df.head()


In [ ]:
# Traz nomes legiveis das tabelas de apoio (facilita MUITO a interpretacao na EDA)
co_pais = pd.read_sql_query("SELECT CO_PAIS, NO_PAIS FROM co_pais;", conn)
co_via  = pd.read_sql_query("SELECT CO_VIA, NO_VIA FROM co_via;", conn)
co_urf  = pd.read_sql_query("SELECT CO_URF, NO_URF FROM co_urf;", conn)
co_unid = pd.read_sql_query("SELECT CO_UNID, NO_UNID FROM co_unid;", conn)
# co_ncm pode ter uma coluna diferente para a descricao do produto -- conferir com
# pd.read_sql_query("PRAGMA table_info(co_ncm);", conn) antes de usar

df = df.merge(co_pais, on="CO_PAIS", how="left")
df = df.merge(co_via, on="CO_VIA", how="left")
df = df.merge(co_urf, on="CO_URF", how="left")
df = df.merge(co_unid, on="CO_UNID", how="left")

df.head()


## 5.1 Identificação e descrição do problema

- **Título:** Estimativa do Valor FOB em Operações de Comércio Exterior do Brasil
- **Integrantes:** [NOME 1], [NOME 2], [NOME 3]
- **Fonte dos dados:** Comex Stat / MDIC, via Kaggle (dataset "Data of Brazilian Import and Export data")
- **Objetivo:** estimar o valor FOB (`VL_FOB`, em US$) de uma operação de comércio exterior a partir de características da transação (produto, país, UF, via de transporte, período etc.), identificando quais fatores mais influenciam o valor negociado.
- **Atributo-alvo:** `VL_FOB` — variável numérica contínua.
- **Atributos preditivos:** `CO_ANO`, `CO_MES`, `CO_NCM`, `CO_PAIS`, `SG_UF_NCM`, `CO_VIA`, `CO_URF`, `QT_ESTAT`, `KG_LIQUIDO`, `TIPO_OPERACAO`.
- **Tipo da tarefa:** **Regressão**, pois o atributo-alvo (`VL_FOB`) representa um valor numérico contínuo, não categorias.


## 5.2 Compreensão dos dados
*(Responsável: Integrante 1)*

In [ ]:
print(df.shape)
df.info()


**Interpretação:** *(preencher após rodar as células acima — não deixar apenas o output bruto; explicar quantidade de registros/atributos, tipos de variável, presença de valores ausentes, duplicatas e inconsistências encontradas).*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df["VL_FOB"], bins=50, ax=axes[0]).set_title("VL_FOB (escala original)")
sns.histplot(np.log1p(df["VL_FOB"]), bins=50, ax=axes[1]).set_title("log(VL_FOB + 1)")
plt.show()

print(df["VL_FOB"].describe())
print("Assimetria (skew):", df["VL_FOB"].skew())


**Interpretação:** *(comentar assimetria da distribuição de VL_FOB e o que isso sugere para o pré-processamento).*

## 5.3 Análise exploratória
*(Responsável: Integrante 1)*

In [ ]:
# Boxplot de VL_FOB por tipo de operacao
fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(data=df, x="TIPO_OPERACAO", y=np.log1p(df["VL_FOB"]), ax=ax)
ax.set_ylabel("log(VL_FOB + 1)")
plt.show()

# Serie temporal do valor total por ano
valor_por_ano = df.groupby("CO_ANO")["VL_FOB"].sum().reset_index()
fig, ax = plt.subplots(figsize=(8,4))
sns.lineplot(data=valor_por_ano, x="CO_ANO", y="VL_FOB", marker="o", ax=ax)
plt.show()

# Top 10 paises por valor total
top_paises = df.groupby("NO_PAIS")["VL_FOB"].sum().sort_values(ascending=False).head(10)
print(top_paises)

# Correlacao entre variaveis numericas
sns.heatmap(df[["QT_ESTAT", "KG_LIQUIDO", "VL_FOB"]].corr(), annot=True, cmap="coolwarm")
plt.show()


**Interpretação:** *(cada gráfico/tabela acima precisa de um parágrafo interpretando o que se observa — sem isso o roteiro não pontua a seção integralmente).*

## 5.4 Pré-processamento
*(Responsável: Integrante 2)*

In [ ]:
# Pipeline / ColumnTransformer — ajustar (fit) SOMENTE no conjunto de treino
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.impute import SimpleImputer


**Justificativa dos tratamentos:** *(para cada tratamento: qual problema foi encontrado → qual tratamento foi aplicado → por que foi escolhido).*

## 5.5 Separação dos dados
*(Responsável: Integrante 2)*

In [ ]:
# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


**Justificativa da proporção utilizada e da estratégia de validação (cross-validation).**

## 5.6 Modelagem
*(Responsável: Integrante 2)*

In [ ]:
# Baseline + Regressão Linear + Árvore de Decisão + Random Forest
# from sklearn.linear_model import LinearRegression
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import cross_val_score


**Comparação dos modelos e justificativa da escolha do modelo final.**

## 5.7 Avaliação e discussão
*(Responsável: Integrante 3)*

In [ ]:
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Aplicar o modelo final UMA ÚNICA VEZ no conjunto de teste reservado


**Discussão:** *(qual modelo teve melhor resultado e por quê; que erros foram observados; limitações; possíveis melhorias futuras).*

## Declaração de uso de Inteligência Artificial
*(preencher no README.md conforme seção 8 do roteiro: ferramenta utilizada, finalidade, parte do trabalho em que foi usada, forma de verificação do conteúdo/código produzido).*
